**<h1>YOLOv3 Training Pipeline</h1>**

**Pipeline:**
1. Convert VisDrone-MOT annotations to YOLO format
2. Auto-generate `data.yaml`
3. Train YOLOv3 with Ultralytics (augmentation handled online during training)
4. Evaluate best checkpoint

**Install dependencies**

In [ ]:
# Uncomment if running for the first time
# !pip install ultralytics torch torchvision opencv-python pyyaml

**<h3>1. VisDrone → YOLO format conversion</h3>**

Converts VisDrone-MOT annotations to YOLO format.
- Filters background class (class 0) and objects smaller than `MIN_BOX` pixels
- Performs train/val split at sequence level (cleaner than per-frame split)
- No manual augmentation here — Ultralytics handles augmentation during training

**VisDrone classes (after -1 offset):**

| ID | Class |
|----|-------|
| 0 | pedestrian |
| 1 | person |
| 2 | car |
| 3 | van |
| 4 | bus |
| 5 | truck |
| 6 | motor |
| 7 | bicycle |
| 8 | awning-tricycle |
| 9 | tricycle |

In [ ]:
import os
import cv2
import random

# ============================================================
# CONFIGURATION
# ============================================================

VISDRONE_ROOT = "route/to/VisDrone-MOT"   # <-- update this path
OUTPUT_ROOT   = "data"

FRAME_SKIP    = 10      # Keep every N-th frame (reduces temporal redundancy)
VAL_SPLIT     = 0.15    # Fraction of SEQUENCES used for validation
IGNORE_CLASS  = 0       # VisDrone class 0 = ignored region (background)
MIN_BOX       = 8       # Discard objects smaller than this (px) in w or h

SEED          = 42
random.seed(SEED)

# VisDrone class names (index 1-10 in dataset → 0-9 in YOLO)
CLASS_NAMES = [
    "pedestrian", "person", "car", "van", "bus",
    "truck", "motor", "bicycle", "awning-tricycle", "tricycle"
]

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

img_train = os.path.join(OUTPUT_ROOT, "images/train")
img_val   = os.path.join(OUTPUT_ROOT, "images/val")
lab_train = os.path.join(OUTPUT_ROOT, "labels/train")
lab_val   = os.path.join(OUTPUT_ROOT, "labels/val")

for p in [img_train, img_val, lab_train, lab_val]:
    os.makedirs(p, exist_ok=True)

# ============================================================
# SEQUENCE-LEVEL TRAIN/VAL SPLIT
# Split by sequence (not by frame) to avoid temporal data leakage.
# Frames from the same video sequence should not appear in both splits.
# ============================================================

sequences_path   = os.path.join(VISDRONE_ROOT, "sequences")
annotations_path = os.path.join(VISDRONE_ROOT, "annotations")

all_sequences = sorted(os.listdir(sequences_path))
random.shuffle(all_sequences)

n_val = max(1, int(len(all_sequences) * VAL_SPLIT))
val_sequences = set(all_sequences[:n_val])
train_sequences = set(all_sequences[n_val:])

print(f"Total sequences : {len(all_sequences)}")
print(f"Train sequences : {len(train_sequences)}")
print(f"Val sequences   : {len(val_sequences)}")

# ============================================================
# CONVERSION LOOP
# ============================================================

image_counter = {"train": 0, "val": 0}

for seq in all_sequences:

    seq_dir  = os.path.join(sequences_path, seq)
    ann_file = os.path.join(annotations_path, seq + ".txt")

    if not os.path.isfile(ann_file):
        print(f"  [SKIP] No annotation file for: {seq}")
        continue

    is_val = seq in val_sequences
    split  = "val" if is_val else "train"
    out_img = img_val   if is_val else img_train
    out_lab = lab_val   if is_val else lab_train

    print(f"[{split.upper()}] Processing: {seq}")

    # ----------------------------------------------------------
    # Parse annotation file
    # Format: frame_id, track_id, x, y, w, h, score, class, truncation, occlusion
    # ----------------------------------------------------------
    annotations = {}

    with open(ann_file) as f:
        for line in f:
            parts = line.strip().split(",")
            if len(parts) < 8:
                continue

            frame_id = int(parts[0])
            x        = float(parts[2])
            y        = float(parts[3])
            w        = float(parts[4])
            h        = float(parts[5])
            cls      = int(parts[7])

            if cls == IGNORE_CLASS:
                continue

            # Shift class index: VisDrone 1-10 → YOLO 0-9
            cls = cls - 1

            if cls < 0 or cls > 9:
                continue

            # Filter very small objects
            if w < MIN_BOX or h < MIN_BOX:
                continue

            annotations.setdefault(frame_id, []).append((x, y, w, h, cls))

    # ----------------------------------------------------------
    # Process frames
    # ----------------------------------------------------------
    frames = sorted(os.listdir(seq_dir))

    for i, frame_name in enumerate(frames):

        if i % FRAME_SKIP != 0:
            continue

        # Parse frame id from filename (e.g. "0000001.jpg" → 1)
        frame_id = int(os.path.splitext(frame_name)[0])

        img_path = os.path.join(seq_dir, frame_name)
        img = cv2.imread(img_path)

        if img is None:
            continue

        H, W = img.shape[:2]
        label_lines = []

        for (x, y, w, h, cls) in annotations.get(frame_id, []):

            # Normalize to YOLO format: xc, yc, w, h relative to image size
            xc     = (x + w / 2) / W
            yc     = (y + h / 2) / H
            w_norm = w / W
            h_norm = h / H

            # Clamp to [0, 1] to handle edge annotations
            xc     = min(max(xc, 0.0), 1.0)
            yc     = min(max(yc, 0.0), 1.0)
            w_norm = min(max(w_norm, 0.0), 1.0)
            h_norm = min(max(h_norm, 0.0), 1.0)

            label_lines.append(f"{cls} {xc:.6f} {yc:.6f} {w_norm:.6f} {h_norm:.6f}")

        if not label_lines:
            continue

        stem = os.path.splitext(frame_name)[0]
        out_img_path = os.path.join(out_img, f"{seq}_{frame_name}")
        out_lab_path = os.path.join(out_lab, f"{seq}_{stem}.txt")

        cv2.imwrite(out_img_path, img)

        with open(out_lab_path, "w") as f:
            f.write("\n".join(label_lines))

        image_counter[split] += 1

# ============================================================
# SUMMARY
# ============================================================
print("\n=== Dataset Generation Complete ===")
print(f"Train images : {image_counter['train']}")
print(f"Val images   : {image_counter['val']}")
print(f"Total        : {sum(image_counter.values())}")

**<h3>2. Generate `data.yaml`</h3>**

Ultralytics requires a YAML file describing the dataset paths and class names.

In [ ]:
import yaml
import os

OUTPUT_ROOT = "data"   # Must match the OUTPUT_ROOT from Cell 2

data_yaml = {
    "path": os.path.abspath(OUTPUT_ROOT),
    "train": "images/train",
    "val":   "images/val",
    "nc":    10,
    "names": [
        "pedestrian", "person", "car", "van", "bus",
        "truck", "motor", "bicycle", "awning-tricycle", "tricycle"
    ]
}

yaml_path = os.path.join(OUTPUT_ROOT, "data.yaml")

with open(yaml_path, "w") as f:
    yaml.dump(data_yaml, f, default_flow_style=False, sort_keys=False)

print(f"data.yaml generated at: {os.path.abspath(yaml_path)}")
print()

# Verify paths exist
for split in ["train", "val"]:
    p = os.path.join(OUTPUT_ROOT, "images", split)
    count = len(os.listdir(p)) if os.path.exists(p) else 0
    status = "✅" if count > 0 else "❌"
    print(f"{status} images/{split}: {count} files")

**GPU Check**

In [ ]:
import torch

def check_device():
    if torch.cuda.is_available():
        name  = torch.cuda.get_device_name(0)
        vram  = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU: {name}  |  VRAM: {vram:.1f} GB")
        return 0
    else:
        print("⚠️  No GPU detected — training will be very slow on CPU.")
        return "cpu"

DEVICE = check_device()

**<h3>3. YOLOv3 Training with Ultralytics augmentation</h3>**

1. **Augmentation strategy (handled online by Ultralytics during training):**

| Augmentation | Value | Reason |
|---|---|---|
| `mosaic` | 1.0 | Most impactful for small object detection |
| `degrees` | 90.0 | Vehicles appear at any angle from a drone |
| `scale` | 0.5 | Simulates different drone altitudes |
| `hsv_v` | 0.4 | Handles shadows, overcast, midday glare |
| `flipud` | 0.5 | Valid for near-nadir aerial views |
| `fliplr` | 0.5 | Standard horizontal flip |
| `copy_paste` | 0.1 | Helps underrepresented classes (pedestrians) |
| `mixup` | 0.05 | Light synthetic mixing |

2. **Key fixes vs. original notebook:**

- `rect: True` removed — incompatible with mosaic (Ultralytics silently disables mosaic when rect is on)
- `imgsz: 640` — appropriate starting point for YOLOv3; push to 832 only if VRAM allows
- `freeze` removed — only used during fine-tuning on a second dataset
- `model: yolov3.pt` — downloads official Ultralytics YOLOv3 weights

In [ ]:
import os
import time
from datetime import datetime
from ultralytics import YOLO

# ============================================================
# MODEL AND OUTPUT CONFIGURATION
# ============================================================

MODEL_WEIGHTS = "yolov3.pt"     # Ultralytics YOLOv3 pretrained on COCO
PROJECT_DIR   = "runs/train_visdrone"
RUN_NAME      = f"yolov3_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
DATA_YAML     = "./data/data.yaml"

# ============================================================
# TRAINING PARAMETERS
# ============================================================
# imgsz=640: safe starting point for YOLOv3.
#   → If your GPU has >10GB VRAM and training is stable, you can increase to 832.
#   → Do NOT use 960 with batch=16 on a mid-range GPU (will OOM).
#
# rect=False (default): required to keep mosaic active.
#   → rect=True disables mosaic silently — a critical loss for small object detection.
#
# freeze: intentionally absent here.
#   → freeze is for fine-tuning on a second dataset (FAC), not initial VisDrone training.

train_params = {
    "data"        : DATA_YAML,
    "epochs"      : 150,
    "patience"    : 25,          # Early stopping: halt if no improvement for 25 epochs
    "imgsz"       : 640,         # Start here; increase to 832 if VRAM allows
    "batch"       : 16,          # Reduce to 8 if you get OOM errors
    "device"      : DEVICE,
    "optimizer"   : "AdamW",
    "lr0"         : 0.001,       # Initial learning rate (slightly higher than yolo11 default)
    "lrf"         : 0.01,        # Final lr = lr0 * lrf (cosine decay)
    "weight_decay": 0.0005,
    "warmup_epochs": 3,

    # ----------------------------------------------------------
    # AUGMENTATION — fully handled by Ultralytics during training
    # No manual augmentation needed in preprocessing.
    # ----------------------------------------------------------

    # Geometric
    "mosaic"      : 1.0,    # Combine 4 images — #1 boost for small object detection
    "close_mosaic": 15,     # Disable mosaic for the last N epochs for stable convergence
    "degrees"     : 90.0,   # Rotation — critical for aerial views (objects at any angle)
    "scale"       : 0.5,    # Zoom in/out — simulates drone altitude changes
    "shear"       : 2.0,    # Slight perspective distortion
    "perspective" : 0.0001, # Small camera angle variation
    "flipud"      : 0.5,    # Vertical flip — valid for nadir aerial imagery
    "fliplr"      : 0.5,    # Horizontal flip
    "translate"   : 0.1,    # Small spatial translation

    # Color / photometric
    "hsv_h"       : 0.015,  # Hue shift (vehicle color variation)
    "hsv_s"       : 0.7,    # Saturation variation
    "hsv_v"       : 0.4,    # Brightness variation — handles shadows and lighting

    # Advanced mixing
    "mixup"       : 0.05,   # Light image blending (keep low for dense scenes)
    "copy_paste"  : 0.1,    # Copy objects across images (good for rare classes)

    # ----------------------------------------------------------
    # OUTPUT AND LOGGING
    # ----------------------------------------------------------
    "project"     : PROJECT_DIR,
    "name"        : RUN_NAME,
    "exist_ok"    : False,
    "plots"       : True,
    "save"        : True,
    "save_period" : 25,      # Save checkpoint every N epochs
    "verbose"     : True,
}

# ============================================================
# ETA CALLBACK
# ============================================================

class ETACallback:
    def __init__(self, total_epochs):
        self.total_epochs = total_epochs
        self.start_time   = None

    def on_train_start(self, trainer):
        self.start_time = time.time()
        print(f"\n🚀 Training started: {RUN_NAME}")
        print(f"   Model     : {MODEL_WEIGHTS}")
        print(f"   Data      : {DATA_YAML}")
        print(f"   Output    : {PROJECT_DIR}/{RUN_NAME}\n")

    def on_fit_epoch_end(self, trainer):
        epoch   = trainer.epoch + 1
        elapsed = time.time() - self.start_time
        eta_sec = (elapsed / epoch) * (self.total_epochs - epoch)
        eta_h   = int(eta_sec // 3600)
        eta_m   = int((eta_sec % 3600) // 60)
        eta_s   = int(eta_sec % 60)
        print(f"⏳ Epoch {epoch}/{self.total_epochs} — ETA: {eta_h}h {eta_m}m {eta_s}s")

# ============================================================
# TRAINING
# ============================================================

def main():
    model  = YOLO(MODEL_WEIGHTS)
    eta_cb = ETACallback(train_params["epochs"])
    model.add_callback("on_train_start",   eta_cb.on_train_start)
    model.add_callback("on_fit_epoch_end", eta_cb.on_fit_epoch_end)

    model.train(**train_params)

if __name__ == "__main__":
    main()

**<h3>4. Evaluate best checkpoint</h3>**

In [ ]:
import os
from ultralytics import YOLO

# Update RUN_NAME if running this cell independently
best_weights = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "best.pt")

if not os.path.exists(best_weights):
    print("⚠️  best.pt not found, falling back to last.pt")
    best_weights = os.path.join(PROJECT_DIR, RUN_NAME, "weights", "last.pt")

print(f"Evaluating: {best_weights}")

model_best = YOLO(best_weights)
metrics    = model_best.val(
    data  = DATA_YAML,
    imgsz = train_params["imgsz"],
    split = "val"
)

print("\n=== Final Metrics ===")
print(f"mAP50     : {metrics.box.map50:.4f}")
print(f"mAP50-95  : {metrics.box.map:.4f}")
print(f"Precision : {metrics.box.mp:.4f}")
print(f"Recall    : {metrics.box.mr:.4f}")
print(f"\nBest model: {best_weights}")

**<h3>5. Fine-tuning on FAC Dataset (Transfer Learning)</h3>**

Run this cell **after** VisDrone training completes.
Update `MODEL_WEIGHTS` with the path to your best VisDrone checkpoint.

In [ ]:
import os
import time
from datetime import datetime
from ultralytics import YOLO

# ← Update this path with your best VisDrone run
MODEL_WEIGHTS = "runs/train_visdrone/YOUR_RUN_NAME/weights/best.pt"

FAC_PROJECT_DIR = "runs/fac_finetune"
FAC_RUN_NAME    = f"visdrone_to_fac_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
FAC_DATA_YAML   = "./datasets/dataset_fac/data.yaml"

finetune_params = {
    "data"          : FAC_DATA_YAML,
    "epochs"        : 70,
    "patience"      : 20,
    "imgsz"         : 640,
    "batch"         : 8,
    "device"        : DEVICE,

    # Fine-tuning learning rate: much lower than initial training
    # Preserves VisDrone features, adapts to FAC domain
    "lr0"           : 0.00005,
    "lrf"           : 0.01,
    "warmup_epochs" : 3,

    # Freeze first 15 layers (backbone) — only train detection head
    # This preserves the low-level aerial features learned from VisDrone
    "freeze"        : 15,

    # Lighter augmentation for fine-tuning
    "mosaic"        : 0.5,
    "close_mosaic"  : 10,
    "flipud"        : 0.5,
    "fliplr"        : 0.5,
    "hsv_v"         : 0.2,
    "mixup"         : 0.0,   # Disabled for fine-tuning

    "project"       : FAC_PROJECT_DIR,
    "name"          : FAC_RUN_NAME,
    "save_period"   : 10,
    "plots"         : True,
}

def main_finetune():
    model  = YOLO(MODEL_WEIGHTS)
    eta_cb = ETACallback(finetune_params["epochs"])
    model.add_callback("on_train_start",   eta_cb.on_train_start)
    model.add_callback("on_fit_epoch_end", eta_cb.on_fit_epoch_end)

    model.train(**finetune_params)

    best = os.path.join(FAC_PROJECT_DIR, FAC_RUN_NAME, "weights", "best.pt")
    if not os.path.exists(best):
        best = os.path.join(FAC_PROJECT_DIR, FAC_RUN_NAME, "weights", "last.pt")

    model_best = YOLO(best)
    metrics    = model_best.val(data=FAC_DATA_YAML, imgsz=640, split="val")

    print("\n=== FAC Fine-tune Metrics ===")
    print(f"mAP50     : {metrics.box.map50:.4f}")
    print(f"mAP50-95  : {metrics.box.map:.4f}")
    print(f"Precision : {metrics.box.mp:.4f}")
    print(f"Recall    : {metrics.box.mr:.4f}")
    print(f"\nBest model: {best}")

if __name__ == "__main__":
    main_finetune()